In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 1. Load files
# ============================================================

model_df_path = Path("../Strategy_4/model_df.csv")
regime_path = Path("orderbook_with_kmeans_regime.csv")
peer_alpha_path = Path("../Strategy_5/peer_alpha_data.csv")

model_df = pd.read_csv(model_df_path)
regime_df = pd.read_csv(regime_path)
peer_alpha_df = pd.read_csv(peer_alpha_path)

print("model_df shape:", model_df.shape)
print("regime_df shape:", regime_df.shape)
print("peer_alpha_df shape:", peer_alpha_df.shape)


# ============================================================
# 2. Reconstruct stock column
# ============================================================

def infer_stock_from_dummies(df):
    df = df.copy()
    df["stock"] = "AMZN"

    if "stock_GOOG" in df.columns:
        df.loc[df["stock_GOOG"] == 1, "stock"] = "GOOG"

    if "stock_INTC" in df.columns:
        df.loc[df["stock_INTC"] == 1, "stock"] = "INTC"

    if "stock_MSFT" in df.columns:
        df.loc[df["stock_MSFT"] == 1, "stock"] = "MSFT"

    return df


model_df = infer_stock_from_dummies(model_df)

print("\nStock distribution:")
print(model_df["stock"].value_counts())


# ============================================================
# 3. Merge K-Means regime labels
# ============================================================

regime_labels = regime_df[
    ["stock", "Time", "OrderID", "regime"]
].copy()

regime_labels = regime_labels.drop_duplicates(
    subset=["stock", "Time", "OrderID"],
    keep="first"
)

model_df = model_df.merge(
    regime_labels,
    on=["stock", "Time", "OrderID"],
    how="left"
)

print("\nAfter regime merge:", model_df.shape)
print("Missing regime:", model_df["regime"].isna().sum())
print("Missing regime pct:", model_df["regime"].isna().mean())

model_df = model_df.dropna(subset=["regime"]).copy()
model_df["regime"] = model_df["regime"].astype(int)

print(model_df["regime"].value_counts().sort_index())


# ============================================================
# 4. Merge peer alpha data
# ============================================================

peer_cols = [
    "stock",
    "Time",
    "OrderID",
    "peer_VWOF",
    "peer_MicroPrice",
    "peer_MicroMomentum",
    "peer_Spread",
    "peer_Spread_Limit",
    "peer_spread_safe",
    "peer_event_frac_in_min",
]

peer_cols = [c for c in peer_cols if c in peer_alpha_df.columns]

peer_alpha_df = peer_alpha_df[peer_cols].copy()

peer_alpha_df = peer_alpha_df.drop_duplicates(
    subset=["stock", "Time", "OrderID"],
    keep="first"
)

model_df = model_df.merge(
    peer_alpha_df,
    on=["stock", "Time", "OrderID"],
    how="left"
)

peer_alpha_cols = [
    "peer_VWOF",
    "peer_MicroPrice",
    "peer_MicroMomentum",
    "peer_Spread",
    "peer_Spread_Limit",
    "peer_spread_safe",
    "peer_event_frac_in_min",
]

peer_alpha_cols = [c for c in peer_alpha_cols if c in model_df.columns]

model_df[peer_alpha_cols] = (
    model_df[peer_alpha_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("\nAfter peer alpha merge:", model_df.shape)
print("Peer alpha columns:", peer_alpha_cols)


# ============================================================
# 5. Define alphas and controls
# ============================================================

your_alpha_cols = [
    "spread",
    "bid_liq_1",
    "ask_liq_1",
    "depth_value_1",
    "imbalance_1",
    "micro_minus_mid",
    "total_bid_size_5",
    "total_ask_size_5",
    "imbalance_5",
    "wap_bid_5",
    "wap_ask_5",
    "mid_chg1",
    "spread_chg1",
    "imbalance_1_chg1",
    "imbalance_5_chg1",
    "micro_minus_mid_chg1",
    "bid_liq_1_chg1",
    "ask_liq_1_chg1",
    "total_bid_size_5_chg1",
    "total_ask_size_5_chg1",
    "event_frac_in_min",
]

your_alpha_cols = [c for c in your_alpha_cols if c in model_df.columns]
peer_alpha_cols = [c for c in peer_alpha_cols if c in model_df.columns]

alpha_features_all = list(dict.fromkeys(your_alpha_cols + peer_alpha_cols))

control_features = [
    "stock_GOOG",
    "stock_INTC",
    "stock_MSFT",
]

control_features = [c for c in control_features if c in model_df.columns]

bool_cols = model_df.select_dtypes(include="bool").columns
model_df[bool_cols] = model_df[bool_cols].astype(int)

print("\nYour alpha count:", len(your_alpha_cols))
print("Peer alpha count:", len(peer_alpha_cols))
print("Total alpha count:", len(alpha_features_all))
print("Controls:", control_features)


# ============================================================
# 6. Alpha significance test by regime
# ============================================================

def test_alpha_significance_by_regime(
    df,
    target_col,
    alpha_features,
    control_features=None,
    min_obs=1000,
    hac_lags=5
):
    if control_features is None:
        control_features = []

    results = []
    feature_cols = alpha_features + control_features

    for regime in sorted(df["regime"].dropna().unique()):

        sub = df[df["regime"] == regime].copy()

        needed_cols = [target_col] + feature_cols
        sub = sub[needed_cols].replace([np.inf, -np.inf], np.nan)

        for col in needed_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        sub = sub.dropna()

        if len(sub) < min_obs:
            print(f"Skipping regime {regime}: only {len(sub)} observations")
            continue

        y = sub[target_col].to_numpy(dtype=float)
        X_df = sub[feature_cols].astype(float)

        non_constant_cols = X_df.columns[X_df.std() > 0].tolist()
        X_df = X_df[non_constant_cols]

        X = sm.add_constant(X_df, has_constant="add").to_numpy(dtype=float)

        fitted = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags}
        )

        param_names = ["const"] + non_constant_cols
        params = pd.Series(fitted.params, index=param_names)
        tvalues = pd.Series(fitted.tvalues, index=param_names)
        pvalues = pd.Series(fitted.pvalues, index=param_names)

        for alpha in alpha_features:

            if alpha in params.index:
                coef = params[alpha]
                tval = tvalues[alpha]
                pval = pvalues[alpha]
                used = True
            else:
                coef = np.nan
                tval = np.nan
                pval = np.nan
                used = False

            if alpha in your_alpha_cols:
                alpha_source = "mine"
            elif alpha in peer_alpha_cols:
                alpha_source = "peer"
            else:
                alpha_source = "unknown"

            results.append({
                "target": target_col,
                "regime": int(regime),
                "alpha_source": alpha_source,
                "alpha": alpha,
                "coef": coef,
                "t_stat": tval,
                "p_value": pval,
                "significant_5pct": bool(pval < 0.05) if not pd.isna(pval) else False,
                "significant_1pct": bool(pval < 0.01) if not pd.isna(pval) else False,
                "helpful": bool((pval < 0.05) and (coef < 0)) if not pd.isna(pval) else False,
                "harmful": bool((pval < 0.05) and (coef > 0)) if not pd.isna(pval) else False,
                "n_obs": int(fitted.nobs),
                "r_squared": fitted.rsquared,
                "used_in_regression": used,
            })

    result_df = pd.DataFrame(results)

    if len(result_df) > 0:
        result_df = result_df.sort_values(
            ["target", "regime", "p_value"],
            na_position="last"
        )

    return result_df


buy_sig = test_alpha_significance_by_regime(
    df=model_df,
    target_col="buy_regret",
    alpha_features=alpha_features_all,
    control_features=control_features,
    min_obs=1000,
    hac_lags=5
)

sell_sig = test_alpha_significance_by_regime(
    df=model_df,
    target_col="sell_regret",
    alpha_features=alpha_features_all,
    control_features=control_features,
    min_obs=1000,
    hac_lags=5
)

sig_results = pd.concat([buy_sig, sell_sig], ignore_index=True)

helpful_sig_alphas = sig_results[
    (sig_results["significant_5pct"] == True) &
    (sig_results["coef"] < 0)
].copy()

harmful_sig_alphas = sig_results[
    (sig_results["significant_5pct"] == True) &
    (sig_results["coef"] > 0)
].copy()

print("\nHelpful + significant alpha count:", len(helpful_sig_alphas))
print("Harmful + significant alpha count:", len(harmful_sig_alphas))

display(
    helpful_sig_alphas[
        ["target", "regime", "alpha_source", "alpha", "coef", "t_stat", "p_value", "n_obs", "r_squared"]
    ].sort_values(["target", "regime", "alpha_source", "p_value"])
)


# ============================================================
# 7. Build regime alpha map
# ============================================================

def build_regime_alpha_map(helpful_sig_alphas):
    regime_alpha_map = {}

    for (target, regime), g in helpful_sig_alphas.groupby(["target", "regime"]):
        alphas = g["alpha"].dropna().unique().tolist()

        if target not in regime_alpha_map:
            regime_alpha_map[target] = {}

        regime_alpha_map[target][int(regime)] = alphas

    return regime_alpha_map


REGIME_ALPHA_MAP = build_regime_alpha_map(helpful_sig_alphas)

print("\nRegime alpha map:")
for target, regime_dict in REGIME_ALPHA_MAP.items():
    print(f"\nTarget: {target}")
    for regime, alphas in regime_dict.items():
        print(f"  Regime {regime}: {alphas}")


# ============================================================
# 8. Prepare dataframe
# ============================================================

def prepare_df_for_regime_strategy(df):
    df = df.copy()

    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(int)

    if "ts" not in df.columns:
        df["ts"] = pd.to_datetime(df["Time"], format="%H:%M:%S.%f", errors="coerce")

    if "minute" not in df.columns:
        df["minute"] = df["ts"].dt.floor("min")

    df = df.dropna(subset=["ts", "minute", "regime"]).copy()
    df["regime"] = df["regime"].astype(int)

    df = df.sort_values(["stock", "ts"]).reset_index(drop=True)

    return df


model_df = prepare_df_for_regime_strategy(model_df)


# ============================================================
# 9. Train/test split by stock-minute
# ============================================================

def train_test_split_by_stock_minute(df, train_frac=0.7):
    train_parts = []
    test_parts = []

    for stock, g in df.groupby("stock"):
        g = g.sort_values("ts").copy()

        unique_minutes = np.array(sorted(g["minute"].unique()))
        split_idx = int(len(unique_minutes) * train_frac)

        train_minutes = unique_minutes[:split_idx]
        test_minutes = unique_minutes[split_idx:]

        train_parts.append(g[g["minute"].isin(train_minutes)])
        test_parts.append(g[g["minute"].isin(test_minutes)])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, test_df


train_df, test_df = train_test_split_by_stock_minute(model_df, train_frac=0.7)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain regime distribution:")
print(train_df["regime"].value_counts().sort_index())

print("\nTest regime distribution:")
print(test_df["regime"].value_counts().sort_index())


# ============================================================
# 10. Train/validation split for threshold tuning
# ============================================================

def split_train_valid_by_stock_minute(df, valid_frac=0.25):
    train_parts = []
    valid_parts = []

    for stock, g in df.groupby("stock"):
        g = g.sort_values("ts").copy()

        unique_minutes = np.array(sorted(g["minute"].unique()))
        split_idx = int(len(unique_minutes) * (1 - valid_frac))

        train_minutes = unique_minutes[:split_idx]
        valid_minutes = unique_minutes[split_idx:]

        train_parts.append(g[g["minute"].isin(train_minutes)])
        valid_parts.append(g[g["minute"].isin(valid_minutes)])

    train_sub = pd.concat(train_parts, ignore_index=True)
    valid_sub = pd.concat(valid_parts, ignore_index=True)

    return train_sub, valid_sub


ridge_fit_df, ridge_valid_df = split_train_valid_by_stock_minute(
    train_df,
    valid_frac=0.25
)

print("\nRidge fit shape:", ridge_fit_df.shape)
print("Ridge valid shape:", ridge_valid_df.shape)


# ============================================================
# 11. Fit one Ridge model per target-regime
# ============================================================

def fit_regime_ridge_models(
    train_df,
    regime_alpha_map,
    control_features,
    ridge_alpha=10.0,
    min_obs=1000
):
    ridge_models = {}
    summaries = []

    for target_col, regime_dict in regime_alpha_map.items():

        for regime, alpha_cols in regime_dict.items():

            alpha_cols = [c for c in alpha_cols if c in train_df.columns]
            controls = [c for c in control_features if c in train_df.columns]

            feature_cols = list(dict.fromkeys(alpha_cols + controls))

            sub = train_df[train_df["regime"] == regime].copy()

            needed_cols = [target_col] + feature_cols
            sub = sub[needed_cols].replace([np.inf, -np.inf], np.nan)

            for col in needed_cols:
                sub[col] = pd.to_numeric(sub[col], errors="coerce")

            sub = sub.dropna()

            if len(sub) < min_obs:
                print(f"Skipping {target_col}, Regime {regime}: only {len(sub)} rows")
                continue

            X = sub[feature_cols]
            y = sub[target_col]

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=ridge_alpha))
            ])

            model.fit(X, y)

            pred_train = model.predict(X)

            ridge_models[(target_col, regime)] = {
                "model": model,
                "alpha_cols": alpha_cols,
                "control_cols": controls,
                "feature_cols": feature_cols,
            }

            summaries.append({
                "target": target_col,
                "regime": regime,
                "n_obs": len(sub),
                "n_alphas": len(alpha_cols),
                "n_controls": len(controls),
                "train_rmse": np.sqrt(mean_squared_error(y, pred_train)),
                "train_r2": r2_score(y, pred_train),
                "alpha_cols": alpha_cols,
                "control_cols": controls,
            })

    return ridge_models, pd.DataFrame(summaries)


ridge_models_v2, ridge_train_summary_v2 = fit_regime_ridge_models(
    train_df=ridge_fit_df,
    regime_alpha_map=REGIME_ALPHA_MAP,
    control_features=control_features,
    ridge_alpha=10.0,
    min_obs=1000
)

display(ridge_train_summary_v2)


# ============================================================
# 12. Add Ridge predictions
# ============================================================

def add_regime_ridge_predictions(df, ridge_models):
    df = df.copy()

    df["pred_buy_regret"] = np.nan
    df["pred_sell_regret"] = np.nan

    for (target_col, regime), obj in ridge_models.items():

        model = obj["model"]
        feature_cols = obj["feature_cols"]

        mask = df["regime"] == regime

        sub = df.loc[mask, feature_cols].copy()
        sub = sub.replace([np.inf, -np.inf], np.nan)

        for col in feature_cols:
            sub[col] = pd.to_numeric(sub[col], errors="coerce")

        valid_idx = sub.dropna().index

        if len(valid_idx) == 0:
            continue

        pred = model.predict(df.loc[valid_idx, feature_cols])

        if target_col == "buy_regret":
            df.loc[valid_idx, "pred_buy_regret"] = pred

        elif target_col == "sell_regret":
            df.loc[valid_idx, "pred_sell_regret"] = pred

    return df


ridge_fit_pred_df = add_regime_ridge_predictions(ridge_fit_df, ridge_models_v2)
ridge_valid_pred_df = add_regime_ridge_predictions(ridge_valid_df, ridge_models_v2)
test_pred_df_v2 = add_regime_ridge_predictions(test_df, ridge_models_v2)


# ============================================================
# 13. Strategy 5 fallback helper
# ============================================================

def choose_strategy5_row(g, side):
    """
    Strategy 5 fallback.

    If Strategy 5 triggers, return that row.
    If not, return first row = TWAP.
    """

    side = side.upper()
    g = g.sort_values("ts").copy()

    twap_row = g.iloc[0]

    for _, row in g.iterrows():

        time_progress = row.get(
            "peer_event_frac_in_min",
            row.get("event_frac_in_min", 0)
        )

        cur_vwof = row.get("peer_VWOF", np.nan)
        cur_mom = row.get("peer_MicroMomentum", np.nan)
        cur_spread = row.get("peer_Spread", row.get("spread", np.nan))
        cur_limit = row.get("peer_Spread_Limit", np.nan)

        if pd.isna(cur_vwof) or pd.isna(cur_mom) or pd.isna(cur_spread) or pd.isna(cur_limit):
            continue

        is_spread_safe = cur_spread <= cur_limit

        if time_progress <= 0.03:
            continue

        if side == "BUY":
            if is_spread_safe and (
                (cur_vwof > 0.2 and cur_mom > 0)
                or (time_progress > 0.9 and cur_vwof > 0)
            ):
                return row, "fallback_strategy5_trigger"

        elif side == "SELL":
            if is_spread_safe and (
                (cur_vwof < -0.2 and cur_mom < 0)
                or (time_progress > 0.8 and cur_vwof < 0)
            ):
                return row, "fallback_strategy5_trigger"

    return twap_row, "fallback_strategy5_no_trigger_twap"


# ============================================================
# 14. Regime Ridge + Strategy 5 fallback execution
# ============================================================

def execute_regime_ridge_with_strategy5_fallback(
    df,
    side,
    thresholds,
    ridge_regimes=(2, 4)
):
    """
    Regime 2/4:
        Try Ridge trigger first.
        If no Ridge trigger, use Strategy 5 fallback.

    Regime 0/1/3:
        Strategy 5 fallback is naturally used because no Ridge trigger exists.

    Benchmark:
        TWAP = first observation of each stock-minute.
    """

    df = df.copy()
    df = df.sort_values(["stock", "minute", "ts"])

    side = side.upper()

    if side == "BUY":
        exec_col = "AskPrice_1"
        pred_col = "pred_buy_regret"
        target_col = "buy_regret"

    elif side == "SELL":
        exec_col = "BidPrice_1"
        pred_col = "pred_sell_regret"
        target_col = "sell_regret"

    else:
        raise ValueError("side must be BUY or SELL")

    records = []

    for (stock, minute), g in df.groupby(["stock", "minute"]):

        g = g.sort_values("ts").copy()

        twap_row = g.iloc[0]
        twap_price = twap_row[exec_col]

        chosen_row = None
        chosen_reason = None

        for _, row in g.iterrows():

            regime = int(row["regime"])

            if regime not in ridge_regimes:
                continue

            key = (target_col, regime)

            if key not in thresholds:
                continue

            pred_regret = row[pred_col]

            if pd.isna(pred_regret):
                continue

            if pred_regret <= thresholds[key]:
                chosen_row = row
                chosen_reason = f"ridge_trigger_regime_{regime}"
                break

        if chosen_row is None:
            chosen_row, chosen_reason = choose_strategy5_row(g, side)

        adaptive_price = chosen_row[exec_col]

        if side == "BUY":
            improvement = twap_price - adaptive_price
        else:
            improvement = adaptive_price - twap_price

        records.append({
            "stock": stock,
            "minute": minute,
            "side": side,
            "twap_price": twap_price,
            "adaptive_price": adaptive_price,
            "improvement": improvement,
            "chosen_time": chosen_row["Time"],
            "chosen_regime": int(chosen_row["regime"]),
            "chosen_reason": chosen_reason,
            "pred_regret": chosen_row[pred_col] if pred_col in chosen_row.index else np.nan,
        })

    return pd.DataFrame(records)


# ============================================================
# 15. Threshold tuning with validation
# ============================================================

def fit_thresholds_by_quantile(train_pred_df, q):
    thresholds = {}

    for target_col, pred_col in [
        ("buy_regret", "pred_buy_regret"),
        ("sell_regret", "pred_sell_regret"),
    ]:
        for regime in [2, 4]:

            sub = train_pred_df[
                (train_pred_df["regime"] == regime) &
                (train_pred_df[pred_col].notna())
            ]

            if len(sub) == 0:
                continue

            thresholds[(target_col, regime)] = sub[pred_col].quantile(q)

    return thresholds


def tune_threshold_quantile_with_s5_fallback(
    train_pred_df,
    valid_pred_df,
    q_grid=np.arange(0.05, 0.85, 0.05)
):
    tuning_results = []

    for q in q_grid:

        thresholds = fit_thresholds_by_quantile(train_pred_df, q=q)

        buy_exec_valid = execute_regime_ridge_with_strategy5_fallback(
            df=valid_pred_df,
            side="BUY",
            thresholds=thresholds,
            ridge_regimes=(2, 4)
        )

        sell_exec_valid = execute_regime_ridge_with_strategy5_fallback(
            df=valid_pred_df,
            side="SELL",
            thresholds=thresholds,
            ridge_regimes=(2, 4)
        )

        valid_exec = pd.concat([buy_exec_valid, sell_exec_valid], ignore_index=True)

        tuning_results.append({
            "q": q,
            "avg_improvement": valid_exec["improvement"].mean(),
            "buy_avg_improvement": buy_exec_valid["improvement"].mean(),
            "sell_avg_improvement": sell_exec_valid["improvement"].mean(),
            "ridge_trigger_rate": (
                valid_exec["chosen_reason"]
                .str.contains("ridge_trigger")
                .mean()
            ),
            "strategy5_trigger_rate": (
                valid_exec["chosen_reason"]
                .str.contains("fallback_strategy5_trigger")
                .mean()
            ),
        })

    tuning_df = pd.DataFrame(tuning_results)

    best_q = tuning_df.sort_values(
        "avg_improvement",
        ascending=False
    ).iloc[0]["q"]

    return best_q, tuning_df


best_q, tuning_df = tune_threshold_quantile_with_s5_fallback(
    train_pred_df=ridge_fit_pred_df,
    valid_pred_df=ridge_valid_pred_df,
    q_grid=np.arange(0.05, 0.85, 0.05)
)

print("\nBest q:", best_q)
display(tuning_df.sort_values("avg_improvement", ascending=False))


# ============================================================
# 16. Run final Regime Ridge + Strategy 5 fallback on test
# ============================================================

best_thresholds = fit_thresholds_by_quantile(
    train_pred_df=ridge_fit_pred_df,
    q=best_q
)

buy_exec_s5fb = execute_regime_ridge_with_strategy5_fallback(
    df=test_pred_df_v2,
    side="BUY",
    thresholds=best_thresholds,
    ridge_regimes=(2, 4)
)

sell_exec_s5fb = execute_regime_ridge_with_strategy5_fallback(
    df=test_pred_df_v2,
    side="SELL",
    thresholds=best_thresholds,
    ridge_regimes=(2, 4)
)

exec_results_s5fb = pd.concat([buy_exec_s5fb, sell_exec_s5fb], ignore_index=True)

print("\nFinal test performance:")
print("Buy avg improvement:", buy_exec_s5fb["improvement"].mean())
print("Sell avg improvement:", sell_exec_s5fb["improvement"].mean())
print("Overall avg improvement:", exec_results_s5fb["improvement"].mean())

display(
    exec_results_s5fb
    .groupby(["side", "chosen_reason"])
    .agg(
        count=("improvement", "size"),
        avg_improvement=("improvement", "mean"),
        total_improvement=("improvement", "sum"),
        win_rate=("improvement", lambda x: (x > 0).mean())
    )
    .reset_index()
)


# ============================================================
# 17. Export final result in backtest format
# ============================================================

regime_ridge_s5fb_buy_mean = (
    exec_results_s5fb[exec_results_s5fb["side"] == "BUY"]
    .groupby("stock", as_index=False)["improvement"]
    .mean()
    .rename(columns={"improvement": "average_improvement"})
)

regime_ridge_s5fb_sell_mean = (
    exec_results_s5fb[exec_results_s5fb["side"] == "SELL"]
    .groupby("stock", as_index=False)["improvement"]
    .mean()
    .rename(columns={"improvement": "average_improvement"})
)

display(regime_ridge_s5fb_buy_mean)
display(regime_ridge_s5fb_sell_mean)

regime_ridge_s5fb_buy_mean.to_csv(
    "regime_ridge_s5fallback_buy_execution.csv",
    index=False
)

regime_ridge_s5fb_sell_mean.to_csv(
    "regime_ridge_s5fallback_sell_execution.csv",
    index=False
)

exec_results_s5fb.to_csv(
    "regime_ridge_s5fallback_execution_detail.csv",
    index=False
)

tuning_df.to_csv(
    "regime_ridge_s5fallback_threshold_tuning.csv",
    index=False
)

print("\nSaved:")
print("regime_ridge_s5fallback_buy_execution.csv")
print("regime_ridge_s5fallback_sell_execution.csv")
print("regime_ridge_s5fallback_execution_detail.csv")
print("regime_ridge_s5fallback_threshold_tuning.csv")

model_df shape: (1149529, 72)
regime_df shape: (1149529, 62)
peer_alpha_df shape: (1149529, 10)

Stock distribution:
stock
MSFT    443425
INTC    430718
AMZN    175338
GOOG    100048
Name: count, dtype: int64

After regime merge: (1149529, 74)
Missing regime: 0
Missing regime pct: 0.0
regime
0    161739
1    763028
2     25000
3    195324
4      4438
Name: count, dtype: int64

After peer alpha merge: (1149529, 81)
Peer alpha columns: ['peer_VWOF', 'peer_MicroPrice', 'peer_MicroMomentum', 'peer_Spread', 'peer_Spread_Limit', 'peer_spread_safe', 'peer_event_frac_in_min']

Your alpha count: 21
Peer alpha count: 7
Total alpha count: 28
Controls: ['stock_GOOG', 'stock_INTC', 'stock_MSFT']

Helpful + significant alpha count: 33
Harmful + significant alpha count: 26


,target,regime,alpha_source,alpha,coef,t_stat,p_value,n_obs,r_squared
57,buy_regret,2,mine,wap_bid_5,-6.891794e-01,-9.784602,1.311096e-22,25000,0.563804
60,buy_regret,2,mine,wap_ask_5,-3.553898e-01,-7.334271,2.229321e-13,25000,0.563804
61,buy_regret,2,mine,micro_minus_mid,-8.704704e-01,-6.787461,1.141244e-11,25000,0.563804
64,buy_regret,2,mine,total_ask_size_5,-1.101736e-06,-5.640827,1.692356e-08,25000,0.563804
65,buy_regret,2,mine,ask_liq_1_chg1,-7.149544e-08,-3.345383,8.216889e-04,25000,0.563804
68,buy_regret,2,mine,total_bid_size_5_chg1,-5.663756e-07,-2.501686,1.236036e-02,25000,0.563804
69,buy_regret,2,mine,bid_liq_1_chg1,-2.389182e-08,-2.408469,1.601957e-02,25000,0.563804
58,buy_regret,2,peer,peer_VWOF,-5.593648e-02,-7.949385,1.874391e-15,25000,0.563804
71,buy_regret,2,peer,peer_Spread,-2.258127e-01,-2.186892,2.875040e-02,25000,0.563804
114,buy_regret,4,mine,wap_bid_5,-6.566298e-01,-8.394896,4.663087e-17,4438,0.614938



Regime alpha map:

Target: buy_regret
  Regime 2: ['wap_bid_5', 'peer_VWOF', 'wap_ask_5', 'micro_minus_mid', 'total_ask_size_5', 'ask_liq_1_chg1', 'total_bid_size_5_chg1', 'bid_liq_1_chg1', 'peer_Spread']
  Regime 4: ['wap_bid_5', 'micro_minus_mid', 'peer_Spread_Limit', 'spread_chg1', 'peer_MicroMomentum', 'ask_liq_1_chg1', 'peer_VWOF', 'imbalance_5_chg1', 'bid_liq_1', 'wap_ask_5']

Target: sell_regret
  Regime 2: ['mid_chg1', 'total_ask_size_5', 'imbalance_5', 'ask_liq_1_chg1', 'event_frac_in_min', 'imbalance_1_chg1']
  Regime 4: ['mid_chg1', 'total_ask_size_5', 'peer_Spread_Limit', 'imbalance_5', 'event_frac_in_min', 'peer_MicroPrice', 'micro_minus_mid_chg1', 'peer_Spread']

Train shape: (883579, 81)
Test shape: (265950, 81)

Train regime distribution:
regime
0    137628
1    565994
2     18877
3    157099
4      3981
Name: count, dtype: int64

Test regime distribution:
regime
0     24111
1    197034
2      6123
3     38225
4       457
Name: count, dtype: int64

Ridge fit shape: (68

,target,regime,n_obs,n_alphas,n_controls,train_rmse,train_r2,alpha_cols,control_cols
0,buy_regret,2,15132,9,3,0.178263,0.491940,"[wap_bid_5, peer_VWOF, wap_ask_5, micro_minus_...","[stock_GOOG, stock_INTC, stock_MSFT]"
1,buy_regret,4,3632,10,3,0.409572,0.302157,"[wap_bid_5, micro_minus_mid, peer_Spread_Limit...","[stock_GOOG, stock_INTC, stock_MSFT]"
2,sell_regret,2,15132,6,3,0.135133,0.213549,"[mid_chg1, total_ask_size_5, imbalance_5, ask_...","[stock_GOOG, stock_INTC, stock_MSFT]"
3,sell_regret,4,3632,8,3,0.240093,0.208252,"[mid_chg1, total_ask_size_5, peer_Spread_Limit...","[stock_GOOG, stock_INTC, stock_MSFT]"



Best q: 0.1


...


Final test performance:
Buy avg improvement: 0.017407407407407597
Sell avg improvement: 0.007932098765431475
Overall avg improvement: 0.012669753086419535


,side,chosen_reason,count,avg_improvement,total_improvement,win_rate
0,BUY,fallback_strategy5_no_trigger_twap,4,0.000000,0.00,0.000000
1,BUY,fallback_strategy5_trigger,86,0.016860,1.45,0.302326
2,BUY,ridge_trigger_regime_2,201,0.026517,5.33,0.467662
3,BUY,ridge_trigger_regime_4,33,-0.034545,-1.14,0.424242
4,SELL,fallback_strategy5_no_trigger_twap,11,0.000000,0.00,0.000000
5,SELL,fallback_strategy5_trigger,171,0.010351,1.77,0.315789
6,SELL,ridge_trigger_regime_2,113,-0.003540,-0.40,0.309735
7,SELL,ridge_trigger_regime_4,29,0.041379,1.20,0.551724


,stock,average_improvement
0,AMZN,0.043086
1,GOOG,0.024815
2,INTC,0.000617
3,MSFT,0.001111


,stock,average_improvement
0,AMZN,0.006173
1,GOOG,0.029753
2,INTC,-0.001358
3,MSFT,-0.002840



Saved:
regime_ridge_s5fallback_buy_execution.csv
regime_ridge_s5fallback_sell_execution.csv
regime_ridge_s5fallback_execution_detail.csv
regime_ridge_s5fallback_threshold_tuning.csv
